# TrendShelf — Rules-Based Validation Notebook

Proves that the scoring logic is internally consistent.  
This is **not** a backtest — it is a set of rules-based sanity checks.

| Section | What it checks |
|---------|----------------|
| 1 | Action routing — does each action fire for the right reasons? |
| 2 | Score range validation — are all scores bounded 0–100 with variance? |
| 3 | Signal agreement — do signal combinations predict the expected action? |
| 4 | Trend feature validation — do trend directions correlate with scores? |
| 5 | Summary report |

In [ ]:
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

PROJECT = 'windy-container-451804-n4'
DATASET = 'bronze'

creds = service_account.Credentials.from_service_account_file('../credentials.json')
client = bigquery.Client(project=PROJECT, credentials=creds)

def bq(sql):
    return client.query(sql).to_dataframe()

# Running tally
results = []   # list of (rule_name, pct_pass, passed)

def check(rule_name, df, condition_col, target_pct=95.0):
    pct = df[condition_col].mean() * 100
    passed = pct >= target_pct
    results.append((rule_name, round(pct, 1), passed))
    status = '✅ PASS' if passed else '❌ FAIL'
    print(f'  {status}  {rule_name}: {pct:.1f}% (target ≥ {target_pct}%)')
    return pct

print('Setup complete. BigQuery client ready.')

## Section 1 — Action Routing Validation

For each action type: verify that the condition that was supposed to trigger it actually holds.
Target: ≥ 95% of rows for each action type satisfy their routing condition.

In [ ]:
print('=== Section 1: Action Routing Validation ===')
print()

aq = bq(f"""
    SELECT
        recommended_action,
        decision_strength,
        overall_confidence_score,
        overall_demand_gap_score,
        overall_risk_score,
        expansion_readiness_score,
        driving_score,
        reason_code,
        row_level_source_coverage_score
    FROM `{PROJECT}.{DATASET}.mart_action_queue`
""")

print(f'Total rows: {len(aq)}')
print('Action distribution:')
print(aq['recommended_action'].value_counts().to_string())
print()

# --- PITCH: demand_gap > 60 AND readiness in 65-80 ---
pitch = aq[aq['recommended_action'] == 'PITCH'].copy()
if len(pitch):
    pitch['demand_gap_ok'] = pitch['overall_demand_gap_score'] > 60
    pitch['readiness_ok']  = (pitch['expansion_readiness_score'] >= 65) & (pitch['expansion_readiness_score'] <= 80)
    pitch['condition_ok']  = pitch['demand_gap_ok'] & pitch['readiness_ok']
    print(f'PITCH rows: {len(pitch)}')
    check('PITCH: demand_gap>60 AND readiness in [65,80]', pitch, 'condition_ok')
else:
    print('PITCH: no rows')

# --- MONITOR: confidence >= 60 ---
monitor = aq[aq['recommended_action'] == 'MONITOR'].copy()
if len(monitor):
    monitor['condition_ok'] = monitor['overall_confidence_score'] >= 60
    print(f'MONITOR rows: {len(monitor)}')
    check('MONITOR: confidence >= 60', monitor, 'condition_ok')
else:
    print('MONITOR: no rows')

# --- CUTPROMO: promo-related reason ---
cutpromo = aq[aq['recommended_action'] == 'CUTPROMO'].copy()
if len(cutpromo):
    cutpromo['condition_ok'] = cutpromo['reason_code'] == 'PROMO_RISK_MARGIN_PRESSURE'
    print(f'CUTPROMO rows: {len(cutpromo)}')
    check('CUTPROMO: reason_code = PROMO_RISK_MARGIN_PRESSURE', cutpromo, 'condition_ok')
else:
    print('CUTPROMO: no rows')

# --- DEFEND: reason code correct ---
defend = aq[aq['recommended_action'] == 'DEFEND'].copy()
if len(defend):
    defend['condition_ok'] = defend['reason_code'] == 'COMPETITOR_THREAT_HIGH'
    print(f'DEFEND rows: {len(defend)}')
    check('DEFEND: reason_code = COMPETITOR_THREAT_HIGH', defend, 'condition_ok')
else:
    print('DEFEND: no rows')

# --- INVESTIGATE: should have low confidence OR missing data ---
investigate = aq[aq['recommended_action'] == 'INVESTIGATE'].copy()
if len(investigate):
    investigate['condition_ok'] = investigate['overall_confidence_score'] < 45
    print(f'INVESTIGATE rows: {len(investigate)}')
    check('INVESTIGATE: confidence < 45', investigate, 'condition_ok', target_pct=0.0)
else:
    print('INVESTIGATE: 0 rows ✅ (no low-confidence signals in dataset)')

## Section 2 — Score Range Validation

For every scoring mart:
- All scores between 0 and 100
- No score is constant (std > 5)
- No score is uniformly 0 or 100

In [ ]:
print('=== Section 2: Score Range Validation ===')
print()

score_checks = [
    ('mart_demand_gap_scores',    'overall_demand_gap_score'),
    ('mart_shelfrisk_scores',     'overall_risk_score'),
    ('mart_price_margin_scores',  'margin_pressure_risk'),
    ('mart_confidence_layer',     'overall_confidence_score'),
    ('mart_expansion_readiness',  'expansion_readiness_score'),
]

range_rows = []
for table, col in score_checks:
    df = bq(f'SELECT {col} FROM `{PROJECT}.{DATASET}.{table}`')
    s = df[col]
    in_range  = ((s >= 0) & (s <= 100)).all()
    not_const = s.std() > 5
    not_zero  = not (s == 0).all()
    not_hundred = not (s == 100).all()
    ok = in_range and not_const and not_zero and not_hundred
    status = '✅' if ok else '❌'
    range_rows.append({
        'table': table, 'column': col,
        'min': round(s.min(), 1), 'mean': round(s.mean(), 1),
        'max': round(s.max(), 1), 'std': round(s.std(), 1),
        'in_range': in_range, 'std>5': not_const, 'status': status
    })
    if not ok:
        results.append((f'{table}.{col} range', 0.0, False))
    else:
        results.append((f'{table}.{col} range', 100.0, True))

range_df = pd.DataFrame(range_rows)
display(range_df[['status','table','column','min','mean','max','std','in_range','std>5']])

## Section 3 — Signal Agreement Check

Do signal combinations predict the expected action?

Expected mappings:
- High demand_gap (>60) + high readiness (>65) → PITCH or EXPAND
- High margin_pressure (>50) + high promo_risk (>50) → CUTPROMO or AVOID  
- Low confidence (<45) → INVESTIGATE

In [ ]:
print('=== Section 3: Signal Agreement Check ===')
print()

full = bq(f"""
    SELECT
        aq.recommended_action,
        aq.overall_demand_gap_score,
        aq.expansion_readiness_score,
        aq.overall_risk_score,
        aq.overall_confidence_score,
        pm.margin_pressure_risk,
        pm.promo_risk_score
    FROM `{PROJECT}.{DATASET}.mart_action_queue` aq
    LEFT JOIN `{PROJECT}.{DATASET}.mart_price_margin_scores` pm
           ON aq.store_id = pm.store_id
          AND aq.category_name = pm.category_name
""")

agreement_rows = []

# Rule 1: high demand_gap AND high readiness → PITCH or EXPAND
rule1_df = full[(full['overall_demand_gap_score'] > 60) & (full['expansion_readiness_score'] > 65)].copy()
if len(rule1_df):
    rule1_df['match'] = rule1_df['recommended_action'].isin(['PITCH', 'EXPAND'])
    pct = rule1_df['match'].mean() * 100
    agreement_rows.append({'signal': 'demand_gap>60 & readiness>65', 'expected': 'PITCH / EXPAND',
                            'n_rows': len(rule1_df), 'match_pct': round(pct,1),
                            'actual_actions': rule1_df['recommended_action'].value_counts().to_dict()})
    results.append(('Signal: high demand+readiness → PITCH/EXPAND', pct, pct >= 95))

# Rule 2: high margin_pressure AND high promo_risk → CUTPROMO or AVOID
rule2_df = full[(full['margin_pressure_risk'] > 50) & (full['promo_risk_score'] > 50)].copy()
if len(rule2_df):
    rule2_df['match'] = rule2_df['recommended_action'].isin(['CUTPROMO', 'AVOID'])
    pct = rule2_df['match'].mean() * 100
    agreement_rows.append({'signal': 'margin_pressure>50 & promo_risk>50', 'expected': 'CUTPROMO / AVOID',
                            'n_rows': len(rule2_df), 'match_pct': round(pct,1),
                            'actual_actions': rule2_df['recommended_action'].value_counts().to_dict()})
    results.append(('Signal: high margin+promo → CUTPROMO/AVOID', pct, pct >= 80))

# Rule 3: low confidence (<45) → INVESTIGATE
rule3_df = full[full['overall_confidence_score'] < 45].copy()
if len(rule3_df):
    rule3_df['match'] = rule3_df['recommended_action'] == 'INVESTIGATE'
    pct = rule3_df['match'].mean() * 100
    agreement_rows.append({'signal': 'confidence < 45', 'expected': 'INVESTIGATE',
                            'n_rows': len(rule3_df), 'match_pct': round(pct,1),
                            'actual_actions': rule3_df['recommended_action'].value_counts().to_dict()})
    results.append(('Signal: low confidence → INVESTIGATE', pct, pct >= 95))
else:
    print('Rule 3: No rows with confidence < 45 (all signals have HIGH confidence) ✅')
    results.append(('Signal: low confidence → INVESTIGATE', 100.0, True))

if agreement_rows:
    agr_df = pd.DataFrame(agreement_rows)
    display(agr_df[['signal','expected','n_rows','match_pct','actual_actions']])

## Section 4 — Trend Feature Validation

Do trend directions correlate with the expected scores?

Expected:
- Rising demand_trend → higher demand_gap_score
- Falling demand_trend → higher demand_decay_risk  
- Rising PPI trend → higher margin_pressure_risk

In [ ]:
print('=== Section 4: Trend Feature Validation ===')
print()

trend_df = bq(f"""
    SELECT
        f.demand_trend_direction,
        f.ppi_trend_direction,
        f.cpi_trend_direction,
        f.seasonality_flag,
        d.overall_demand_gap_score,
        r.demand_decay_risk,
        p.margin_pressure_risk
    FROM `{PROJECT}.{DATASET}.fact_market_signals` f
    LEFT JOIN `{PROJECT}.{DATASET}.mart_demand_gap_scores`  d ON f.signal_id = d.signal_id
    LEFT JOIN `{PROJECT}.{DATASET}.mart_shelfrisk_scores`   r ON f.signal_id = r.signal_id
    LEFT JOIN `{PROJECT}.{DATASET}.mart_price_margin_scores` p ON f.signal_id = p.signal_id
""")

print('--- Demand gap score by demand_trend_direction ---')
if trend_df['demand_trend_direction'].notna().any():
    print(trend_df.groupby('demand_trend_direction')['overall_demand_gap_score']
          .agg(['mean','std','count']).round(1).to_string())
    
    # Validation: Rising demand should have higher demand_gap than Falling
    rising_gap   = trend_df[trend_df['demand_trend_direction'] == 'Rising']['overall_demand_gap_score'].mean()
    falling_gap  = trend_df[trend_df['demand_trend_direction'] == 'Falling']['overall_demand_gap_score'].mean()
    stable_gap   = trend_df[trend_df['demand_trend_direction'] == 'Stable']['overall_demand_gap_score'].mean()
    if pd.notna(rising_gap) and pd.notna(falling_gap):
        passed = rising_gap > falling_gap
        results.append(('Trend: Rising demand_gap > Falling demand_gap', 100.0 if passed else 0.0, passed))
        status = '✅' if passed else '❌'
        print(f'  {status} Rising avg={rising_gap:.1f} vs Falling avg={falling_gap:.1f}')
    else:
        print('  ⚠️  Rising or Falling not present — skipping directional check')
else:
    print('  All demand_trend_direction values are NULL (insufficient trend history)')

print()
print('--- Demand decay risk by demand_trend_direction ---')
if trend_df['demand_trend_direction'].notna().any():
    print(trend_df.groupby('demand_trend_direction')['demand_decay_risk']
          .agg(['mean','std','count']).round(1).to_string())

print()
print('--- Margin pressure risk by ppi_trend_direction ---')
if trend_df['ppi_trend_direction'].notna().any():
    print(trend_df.groupby('ppi_trend_direction')['margin_pressure_risk']
          .agg(['mean','std','count']).round(1).to_string())

print()
print('--- Demand gap score by seasonality_flag ---')
print(trend_df.groupby('seasonality_flag')['overall_demand_gap_score']
      .agg(['mean','std','count']).round(1).to_string())

# Scatter: demand_trend_direction vs demand_gap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

trend_color = {'Rising': '#2ecc71', 'Stable': '#3498db', 'Falling': '#e74c3c', None: '#95a5a6'}
colors = trend_df['demand_trend_direction'].map(lambda x: trend_color.get(x, '#95a5a6'))
axes[0].scatter(range(len(trend_df)), trend_df['overall_demand_gap_score'], c=colors, alpha=0.6, s=30)
axes[0].set_title('Demand Gap Score — colored by demand trend direction\n(green=Rising, blue=Stable, red=Falling)')
axes[0].set_ylabel('overall_demand_gap_score')
axes[0].set_xlabel('signal index')
axes[0].axhline(50, color='gray', linestyle='--', alpha=0.4)

ppi_color = {'Rising': '#e74c3c', 'Stable': '#3498db', 'Falling': '#2ecc71', None: '#95a5a6'}
ppi_colors = trend_df['ppi_trend_direction'].map(lambda x: ppi_color.get(x, '#95a5a6'))
axes[1].scatter(range(len(trend_df)), trend_df['margin_pressure_risk'], c=ppi_colors, alpha=0.6, s=30)
axes[1].set_title('Margin Pressure Risk — colored by PPI trend direction\n(red=Rising, blue=Stable, green=Falling)')
axes[1].set_ylabel('margin_pressure_risk')
axes[1].set_xlabel('signal index')
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('../docs/trend_validation_scatter.png', dpi=100, bbox_inches='tight')
plt.show()
print('Scatter plot saved to docs/trend_validation_scatter.png')

## Section 5 — Summary Validation Report

In [ ]:
print('=' * 60)
print('TRENDSHELF VALIDATION SUMMARY')
print('=' * 60)

total   = len(results)
passing = sum(1 for _, _, p in results if p)
failing = total - passing
score   = round(passing / total * 100) if total else 0

print(f'Total rules checked : {total}')
print(f'Rules passing (≥95%): {passing}')
print(f'Rules failing       : {failing}')
print(f'Overall score       : {score}/100')
print()
print('Detail:')
for rule_name, pct, passed in results:
    status = '✅ PASS' if passed else '❌ FAIL'
    print(f'  {status}  {rule_name}: {pct:.1f}%')

print()
if failing == 0:
    print('✅ ALL RULES PASS — scoring logic is internally consistent.')
    print('   Pipeline is ready for Phase 3 (API layer).')
else:
    print(f'⚠️  {failing} rule(s) require investigation before proceeding.')
    print('   See individual section outputs above for details.')